# 🔍 CodeReview — AI Code Review Environment

**Meta × PyTorch OpenEnv Hackathon 2026**

An OpenEnv RL environment where an AI agent reviews Python code snippets for bugs, style issues, and security vulnerabilities.

## Environment Overview
| Property | Value |
|----------|-------|
| Action | List of identified issues + summary |
| Observation | Code snippet, description, difficulty, feedback |
| Reward | 0.0 - 1.0 (precision/recall of found issues) |
| Tasks | 6 tasks across 3 difficulty levels |
| Steps per episode | 1 (single review submission) |

## Step 1: Install Dependencies

In [ ]:
!pip install openenv-core fastapi uvicorn openai pyyaml -q

## Step 2: Set Environment Variables
Set your API key below. We use Groq's free LLaMA 3.3 70B model.

In [ ]:
import os

# Set your API credentials here
os.environ["OPENAI_API_KEY"] = "your_api_key_here"  # Replace with your Groq API key
os.environ["API_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["MODEL_NAME"] = "llama-3.3-70b-versatile"
os.environ["ENV_URL"] = "http://localhost:8000"

print("✅ Environment variables set!")

## Step 3: Start the Environment Server
Run this in a **separate terminal**:
```bash
uvicorn code_review_env.server.app:app --host 0.0.0.0 --port 8000
```

## Step 4: Explore the Task Bank
Let's inspect the 6 code review tasks our agent will solve.

In [ ]:
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

from code_review_env.tasks.task_bank import TASKS

print(f"Total tasks: {len(TASKS)}\n")
for task in TASKS:
    print(f"📋 {task['task_id']} ({task['difficulty']}) — {task['description']}")
    print(f"   Hidden issues: {len(task['ground_truth_issues'])}")
    print()

## Step 5: Inspect the Grader Logic
The grader uses **fuzzy matching** on category (exact match) and description keywords (30% overlap).

In [ ]:
from code_review_env.server.grader import grade_review

# Example: Test grader with a sample submission
sample_submission = [
    {"line": "6", "severity": "high", "category": "logic", "description": "Division by zero when the input list is empty"}
]
ground_truth = TASKS[0]["ground_truth_issues"]

reward, feedback, tp, fp = grade_review(sample_submission, ground_truth, "easy")
print(f"Reward: {reward}")
print(f"Feedback: {feedback}")
print(f"True Positives: {tp}, False Positives: {fp}")

## Step 6: The AI Agent — 17-Point Security Review Checklist

Our inference agent uses a **structured prompt** with a mandatory review checklist covering:

1. Division by zero
2. Resource management (file handles)
3. Error handling
4. Off-by-one / Index errors
5. Integer overflow
6. Type hints / Style
7. Kwargs handling
8. Memory leaks (unbounded collections)
9. Unhashable types
10. Weak cryptography (MD5)
11. Timing attacks
12. Authorization gaps
13. Thread safety
14. Performance inefficiencies
15. Stale data
16. Hardcoded secrets
17. Missing persistence

This checklist ensures the LLM systematically evaluates every category of bug the grader checks for.

## Step 7: Run Full Inference
⚠️ Make sure the server is running in a separate terminal first!

In [ ]:
# Run the full inference pipeline
!python inference.py

## 🏆 Results

Our agent consistently achieves the following scores:

| Task | Difficulty | Reward |
|------|-----------|--------|
| easy_001 | Easy | 0.50 – 1.00 |
| easy_002 | Easy | **1.00** ✅ |
| medium_001 | Medium | **1.00** ✅ |
| medium_002 | Medium | 0.33 – 0.67 |
| hard_001 | Hard | 0.60 – 0.80 |
| hard_002 | Hard | 0.25 – 0.50 |
| **Average** | | **0.65 – 0.74** |

### Key Technical Achievements
- ✅ **Perfect 1.00** scores on `easy_002` and `medium_001` consistently
- ✅ Successfully detects **MD5 crypto flaws**, **timing attacks**, and **authorization gaps**
- ✅ Robust JSON parsing with regex extraction + YAML fallback
- ✅ Automatic rate-limit retry with 60-second backoff
- ✅ Category-aligned prompting for precise grader matching

## 🛠️ Tech Stack

- **Framework**: FastAPI + WebSockets (via `openenv-core`)
- **Models**: Pydantic v2 for Action, Observation, State schemas
- **LLM**: Groq (LLaMA 3.3 70B) via OpenAI-compatible API
- **Containerization**: Docker
- **Deployment**: Hugging Face Spaces